In [11]:
# Using the XGBoost Classifier approach
# -------------------------------------------------------------
# Import necessary libraries

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from xgboost import XGBClassifier

COST_1, COST_2 = 10, 500         # challenge metric
SEED = 42                        # reproducibility

# -------------------------------------------------------------
# 1. LOAD CSVs  (skip 20-line GPL header, turn "na" → NaN)
# -------------------------------------------------------------
train_df = pd.read_csv(
    "/Users/minkush/Desktop/aps+failure+at+scania+trucks/aps_failure_training_set.csv",
    na_values="na",
    skiprows=20                  # skip GPL header
)
test_df = pd.read_csv(
    "/Users/minkush/Desktop/aps+failure+at+scania+trucks/aps_failure_test_set.csv",
    na_values="na",
    skiprows=20
)

# Map labels: 'neg'→0, 'pos'→1
train_df["class"] = train_df["class"].map({"neg": 0, "pos": 1})
test_df["class"]  = test_df["class"].map({"neg": 0, "pos": 1})

X = train_df.drop(columns="class")
y = train_df["class"].values
X_test  = test_df.drop(columns="class")
y_test  = test_df["class"].values

# -------------------------------------------------------------
# 2. TRAIN / VALIDATION SPLIT  (80 % / 20 % stratified)
# -------------------------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

# -------------------------------------------------------------
# 3. PREPROCESSOR + XGB MODEL
# -------------------------------------------------------------
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

xgb_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf",  XGBClassifier(
        n_estimators=700,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum(),
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1
    ))
])

print("\nTraining XGBoost … (≈1–2 min)")
xgb_pipeline.fit(X_tr, y_tr)

# -------------------------------------------------------------
# 4. COST FUNCTION + THRESHOLD SEARCH
# -------------------------------------------------------------
def total_cost(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fp, fn = cm[1], cm[2]     # cm layout: [tn fp fn tp]
    return COST_1 * fp + COST_2 * fn

val_proba = xgb_pipeline.predict_proba(X_val)[:, 1]
thresholds = np.linspace(0, 1, 101)               # 0.00 … 1.00
costs = [total_cost(y_val, (val_proba >= t)) for t in thresholds]
tau_star = thresholds[int(np.argmin(costs))]
print(f"\nOptimal threshold τ* = {tau_star:.4f}")

# -------------------------------------------------------------
# 5. REPORTING FUNCTION
# -------------------------------------------------------------
def print_report(name, y_true, y_prob, tau):
    y_pred = (y_prob >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    cost = COST_1 * fp + COST_2 * fn
    print(f"\n### {name} ###")
    print("Confusion matrix (rows=true, cols=pred):")
    print(f"          Pred-Neg   Pred-Pos")
    print(f"True-Neg  {tn:8d}  {fp:8d}")
    print(f"True-Pos  {fn:8d}  {tp:8d}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
    print(f"Total cost: {cost:,}")

# -------------------------------------------------------------
# 6. FINAL METRICS
# -------------------------------------------------------------
print_report("Validation", y_val, val_proba, tau_star)

test_proba = xgb_pipeline.predict_proba(X_test)[:, 1]
print_report("Test", y_test, test_proba, tau_star)


Training XGBoost … (≈1–2 min)

Optimal threshold τ* = 0.0200

### Validation ###
Confusion matrix (rows=true, cols=pred):
          Pred-Neg   Pred-Pos
True-Neg     11522       278
True-Pos         5       195
Precision: 0.412
Recall:    0.975
Total cost: 5,280

### Test ###
Confusion matrix (rows=true, cols=pred):
          Pred-Neg   Pred-Pos
True-Neg     15290       335
True-Pos        16       359
Precision: 0.517
Recall:    0.957
Total cost: 11,350
